# Context-Aware Variants — Matched 5-Seed Observation-Level Comparison

**Purpose:** run only the two new models: (1) context-aware Standard QML and (2) context-aware PAQFE-v2. The uploaded baseline notebooks are the reference for matching the page-14 protocol: one fixed 80/20 observation-level split (`random_state=42`), seeds 42–46, 10 qubits, 3 layers, 300 epochs, Adam, and train-split-only scaling.

**Important:** this reproduces the *observation-level* comparison from page 14; repeated compositions can occur in both train and test. Treat it as a matched diagnostic, not as evidence of generalization to unseen compositions. Use the separate GroupKFold experiment for that claim.

Context-aware mechanism: trainable log-gains for the physics-descriptor group and experimental-context group. Their exponentials multiply standardized features. A small penalty keeps both log-gains near zero (initial gain = 1). Context-aware PAQFE-v2 additionally retains the trainable per-feature physics scaling and its prior penalty.


In [ ]:
!pip -q install pennylane openpyxl


In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEEDS = [42, 43, 44, 45, 46]
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
N_QUBITS = 10
N_LAYERS = 3
LR_VQC = 0.02
LR_ALPHA = 0.005
N_EPOCHS = 300
LAMBDA_PRIOR = 0.01
LAMBDA_GROUP_GAIN = 0.01
UPWEIGHT_FACTOR = 1.2
TARGET = 'UTS_MPa'
STANDARD_FEATURE_ORDER = ['xAl','VEC','dHmix','dSmix','delta','Omega','dchi','Mean_Tm','Temperature_C','log10_Strain_rate']
GROUPED_FEATURE_ORDER = ['xAl','VEC','dchi','dHmix','dSmix','Omega','Mean_Tm','delta','Temperature_C','log10_Strain_rate']
STRONG_RELATIONSHIP_DESCRIPTORS = {'delta','Omega','xAl','VEC'}
COMPOSITION_COLS = ['Al_norm','Co_norm','Cr_norm','Fe_norm','Ni_norm']
PHYSICS_FEATURES = {'xAl','VEC','dHmix','dSmix','delta','Omega','dchi','Mean_Tm'}
CONTEXT_FEATURES = {'Temperature_C','log10_Strain_rate'}

print('Torch:', torch.__version__)
print('PennyLane:', qml.__version__)


## 1. Upload the same workbook
Select the Excel file used for the baseline notebooks. The code reads `B_Model_Ready` and builds the same rounded-composition group identifier for auditing, although this observation-level split does not use groups for splitting.


In [ ]:
from google.colab import files
uploaded = files.upload()
file_path = next(iter(uploaded))
df = pd.read_excel(file_path, sheet_name='B_Model_Ready')
required = list(dict.fromkeys(STANDARD_FEATURE_ORDER + [TARGET] + COMPOSITION_COLS))
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
df[required] = df[required].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=required).copy().reset_index(drop=True)
df['Composition_Group'] = df[COMPOSITION_COLS].round(3).astype(str).agg('_'.join, axis=1)
print('Observations:', len(df))
print('Unique compositions:', df['Composition_Group'].nunique())
print('Rows per composition:', df['Composition_Group'].value_counts().sort_values().to_dict())


## 2. Fixed split and fold-independent preprocessing
This is deliberately the same fixed observation-level split used by the reference stability notebooks. The scalers are fitted only on the training partition.


In [ ]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE)
y_train_raw = train_df[TARGET].to_numpy(dtype=np.float64)
y_test_raw = test_df[TARGET].to_numpy(dtype=np.float64)
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train_raw.reshape(-1, 1)).ravel()

# Fit a separate feature scaler for each feature ordering, on training data only.
scalers, X_train_by_order, X_test_by_order = {}, {}, {}
for order_name, feature_order in [('standard', STANDARD_FEATURE_ORDER), ('grouped', GROUPED_FEATURE_ORDER)]:
    scaler = StandardScaler()
    Xtr_raw = train_df[feature_order].to_numpy(dtype=np.float64)
    Xte_raw = test_df[feature_order].to_numpy(dtype=np.float64)
    X_train_by_order[order_name] = scaler.fit_transform(Xtr_raw)
    X_test_by_order[order_name] = scaler.transform(Xte_raw)
    scalers[order_name] = scaler

print(f'Train observations: {len(train_df)} | Test observations: {len(test_df)}')
print('Train/test composition overlap:', len(set(train_df['Composition_Group']) & set(test_df['Composition_Group'])))
XTR = {k: torch.tensor(v, dtype=torch.float64) for k, v in X_train_by_order.items()}
XTE = {k: torch.tensor(v, dtype=torch.float64) for k, v in X_test_by_order.items()}
YTR = torch.tensor(y_train, dtype=torch.float64)


## 3. Shared quantum circuit and the two context-aware models
Both models use 10 qubits, Y-angle embedding, 3 StronglyEntanglingLayers, Pauli-Z expectation values and a linear readout. The PAQFE-v2 variant adds the same physics-aware entanglement links and trainable feature-wise `alpha` prior as the reference notebook.


In [ ]:
QUBIT_BLOCKS = {'composition':[0,1,2], 'thermodynamic':[3,4,5,6], 'structural':[7], 'context':[8,9]}
physics_prior = torch.tensor([UPWEIGHT_FACTOR if f in STRONG_RELATIONSHIP_DESCRIPTORS else 1.0 for f in GROUPED_FEATURE_ORDER], dtype=torch.float64)

def ring_entangle(wires):
    if len(wires) < 2:
        return
    for i in range(len(wires)):
        qml.CNOT(wires=[wires[i], wires[(i+1) % len(wires)]])

def physics_entangling_layer():
    ring_entangle(QUBIT_BLOCKS['composition'])
    ring_entangle(QUBIT_BLOCKS['thermodynamic'])
    ring_entangle(QUBIT_BLOCKS['context'])
    fmap = {f:i for i, f in enumerate(GROUPED_FEATURE_ORDER)}
    qml.CNOT(wires=[fmap['delta'], fmap['Omega']])
    qml.CNOT(wires=[fmap['xAl'], fmap['VEC']])

weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)

def build_circuit(physics_aware=False):
    dev = qml.device('default.qubit', wires=N_QUBITS)
    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def circuit(inputs, weights):
        qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
        if physics_aware:
            physics_entangling_layer()
        qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]
    return circuit

class ContextAwareStandardQML(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_physics_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
        self.log_context_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
        self.weights = nn.Parameter(0.01 * torch.randn(weight_shape, dtype=torch.float64))
        self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)
        self.circuit = build_circuit(physics_aware=False)
        self.feature_order = STANDARD_FEATURE_ORDER
    def forward(self, x):
        gains = torch.ones(len(self.feature_order), dtype=x.dtype, device=x.device)
        pidx = [i for i,f in enumerate(self.feature_order) if f in PHYSICS_FEATURES]
        cidx = [i for i,f in enumerate(self.feature_order) if f in CONTEXT_FEATURES]
        gains[pidx] = torch.exp(self.log_physics_gain)
        gains[cidx] = torch.exp(self.log_context_gain)
        q = torch.stack(self.circuit(x * gains, self.weights), dim=1)
        return self.head(q).squeeze(-1)
    def gain_penalty(self):
        return LAMBDA_GROUP_GAIN * (self.log_physics_gain.square() + self.log_context_gain.square())

class ContextAwarePAQFEv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(physics_prior.clone())
        self.log_physics_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
        self.log_context_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
        self.weights = nn.Parameter(0.01 * torch.randn(weight_shape, dtype=torch.float64))
        self.head = nn.Linear(N_QUBITS, 1, dtype=torch.float64)
        self.circuit = build_circuit(physics_aware=True)
        self.feature_order = GROUPED_FEATURE_ORDER
    def forward(self, x):
        gains = torch.ones(len(self.feature_order), dtype=x.dtype, device=x.device)
        pidx = [i for i,f in enumerate(self.feature_order) if f in PHYSICS_FEATURES]
        cidx = [i for i,f in enumerate(self.feature_order) if f in CONTEXT_FEATURES]
        gains[pidx] = torch.exp(self.log_physics_gain)
        gains[cidx] = torch.exp(self.log_context_gain)
        theta = x * gains * self.alpha
        q = torch.stack(self.circuit(theta, self.weights), dim=1)
        return self.head(q).squeeze(-1)
    def gain_penalty(self):
        return LAMBDA_GROUP_GAIN * (self.log_physics_gain.square() + self.log_context_gain.square())
    def alpha_penalty(self):
        return LAMBDA_PRIOR * torch.mean((self.alpha - physics_prior) ** 2)

print('Models defined: ContextAwareStandardQML, ContextAwarePAQFEv2')


## 4. Run only the two new variants across seeds 42–46
To match the reference PAQFE-v2 optimizer, `alpha` uses learning rate 0.005 and the circuit weights/readout use 0.02. The context gains use 0.02. Both models use the same split, same training target scaling and same epoch budget. Results are printed once per seed and saved at the end.


In [ ]:
def fit_and_evaluate(model_name, seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if model_name == 'Context-aware Standard QML':
        model = ContextAwareStandardQML()
        optimizer = torch.optim.Adam([
            {'params': [model.log_physics_gain, model.log_context_gain], 'lr': LR_VQC},
            {'params': [model.weights], 'lr': LR_VQC},
            {'params': model.head.parameters(), 'lr': LR_VQC},
        ])
        order_key = 'standard'
    elif model_name == 'Context-aware PAQFE-v2':
        model = ContextAwarePAQFEv2()
        optimizer = torch.optim.Adam([
            {'params': [model.alpha], 'lr': LR_ALPHA},
            {'params': [model.log_physics_gain, model.log_context_gain], 'lr': LR_VQC},
            {'params': [model.weights], 'lr': LR_VQC},
            {'params': model.head.parameters(), 'lr': LR_VQC},
        ])
        order_key = 'grouped'
    else:
        raise ValueError(model_name)

    loss_fn = nn.MSELoss()
    xtr, xte = XTR[order_key], XTE[order_key]
    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        pred = model(xtr)
        loss = loss_fn(pred, YTR) + model.gain_penalty()
        if model_name == 'Context-aware PAQFE-v2':
            loss = loss + model.alpha_penalty()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_std = model(xte).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_std.reshape(-1, 1)).ravel()
    row = {
        'Model': model_name, 'Seed': seed,
        'MAE': mean_absolute_error(y_test_raw, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_raw, pred)),
        'R2': r2_score(y_test_raw, pred),
        'Physics_gain': float(torch.exp(model.log_physics_gain).detach()),
        'Context_gain': float(torch.exp(model.log_context_gain).detach()),
    }
    if model_name == 'Context-aware PAQFE-v2':
        for f, val in zip(GROUPED_FEATURE_ORDER, model.alpha.detach().cpu().numpy()):
            row[f'alpha_{f}'] = float(val)
    return row

# Train only the two new context-aware variants; baselines are not retrained here.
results = []
for model_name in ['Context-aware Standard QML', 'Context-aware PAQFE-v2']:
    for seed in SEEDS:
        row = fit_and_evaluate(model_name, seed)
        results.append(row)
        print(f"Completed: {row['Model']} | seed={seed}")

results_df = pd.DataFrame(results)

# Conference-style summary: one row per model, with mean ± sample SD across seeds.
summary_rows = []
for model_name in ['Context-aware Standard QML', 'Context-aware PAQFE-v2']:
    g = results_df[results_df['Model'] == model_name]
    summary_rows.append({
        'Model': model_name,
        'MAE (mean ± SD)': f"{g['MAE'].mean():.2f} ± {g['MAE'].std(ddof=1):.2f}",
        'RMSE (mean ± SD)': f"{g['RMSE'].mean():.2f} ± {g['RMSE'].std(ddof=1):.2f}",
        'R² (mean ± SD)': f"{g['R2'].mean():.3f} ± {g['R2'].std(ddof=1):.3f}",
    })
conference_summary_df = pd.DataFrame(summary_rows)

print('\n=== MAIN TABLE: Mean ± sample SD across seeds ===')
display(conference_summary_df)
print('\n=== Per-seed metrics (for transparency) ===')
display(results_df[['Model', 'Seed', 'MAE', 'RMSE', 'R2']].round(4))
print('\n=== Learned gains ===')
display(results_df[['Model', 'Seed', 'Physics_gain', 'Context_gain']].round(4))

# Save both paper-ready summary and full per-seed detail.
results_df.to_csv('context_aware_observation_level_seed_results.csv', index=False)
conference_summary_df.to_csv('context_aware_observation_level_summary_paper_table.csv', index=False)
print('Saved: context_aware_observation_level_seed_results.csv')
print('Saved: context_aware_observation_level_summary_paper_table.csv')


## Interpretation checklist
- Compare these two new variants with the existing Standard QML and PAQFE-v2 five-seed results only if the baseline notebooks used the same workbook, same 80/20 split, same preprocessing, same epoch count and same seed policy.
- A fixed test split reused across five initialization seeds measures initialization/training variability, **not** variability across different train/test samples.
- Because this is observation-level splitting, composition overlap can inflate apparent generalization. Use the repeated composition-level GroupKFold experiment as the main evidence for unseen-composition claims.
- Report all metrics, not only the metric that favors the proposed model. Do not claim a win unless differences are consistent and appropriately qualified.
